# Ewaluacja — jak zmierzyć, czy apelacja jest dobra?

Wygenerować apelację to jedno — ale skąd wiadomo, że jest **dobra**? Tu mierzymy to liczbami, zadając **dwa niezależne pytania**:

1. **Czy porusza to, co powinna?** → *pokrycie* (`coverage`, `repeat`)
2. **Czy czegoś nie zmyśliła?** → *ugruntowanie / halucynacje* (`grounding`)

Klucz oceny to `data/eval.json` — lista zagadnień, które dobra apelacja powinna podnieść.

> Szczegóły każdego modułu: [`src/eval/README.md`](../src/eval/README.md).

## 0. Konfiguracja modelu

Domyślnie lokalnie przez Ollamę (`qwen2.5:14b`). Aby użyć OpenAI — odkomentuj sekcję A i zakomentuj B.
Uruchamiaj komórki **po kolei**; po zmianie modelu zrestartuj kernel.

In [ ]:
import os

# ── B. Lokalnie przez Ollamę (domyślnie) ──
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"]  = "ollama"        # placeholder — Ollama nie wymaga klucza
os.environ["LLM_MODEL"]    = "qwen2.5:14b"   # najpierw: ollama pull qwen2.5:14b

# ── A. Własny klucz API (OpenAI) — odkomentuj, zakomentuj sekcję B ──
# os.environ["LLM_BASE_URL"] = "https://api.openai.com/v1"
# os.environ["LLM_API_KEY"]  = "sk-...wpisz-swój-klucz..."
# os.environ["LLM_MODEL"]    = "gpt-4o-mini"

print("model:", os.environ["LLM_MODEL"])

## 1. Wygeneruj apelację do oceny

Bierzemy najprostsze podejście (`baseline`) — wszystkie akta w jeden prompt. Wynik posłuży za materiał do obu ewaluacji poniżej.

In [ ]:
from src.loader import load_all
from baseline.main import generate_appeal

docs = load_all()
print(f"Wczytano {len(docs)} dokumentów")

appeal = generate_appeal(docs).tekst
print(f"Apelacja: {len(appeal):,} znaków\n")
print(appeal[:800], "...")

## 2. Pokrycie — czy porusza wymagane zagadnienia?

Sędzia-LLM ocenia dla **każdego** zagadnienia z klucza, czy apelacja je porusza. Wynik to `covered / total`.

In [ ]:
from src.eval.coverage import evaluate

cov = evaluate(appeal)
print(f"Pokrycie: {cov.covered}/{cov.total} = {cov.score:.0%}\n")

print("Niepokryte zagadnienia:")
for r in cov.results:
    if not r.covered:
        print(f"  • {r.issue[:90]}")

## 3. Powtarzalność — jeden przebieg to za mało

Ten sam prompt za każdym razem daje nieco inną apelację, więc i pokrycie się waha. Dopiero **średnia z kilku przebiegów** mówi coś wiarygodnego.

> ⚠️ Wolne: każdy przebieg = 1 generacja + ocena. Zmniejsz `runs`, jeśli trzeba.

In [ ]:
from src.eval.repeat import repeat

reports = repeat(approach="baseline", runs=3)
# dostępne sposoby: baseline, agent_linear, agent_planner

## 4. Halucynacje — czy apelacja nie zmyśla faktów?

Drugie, niezależne ryzyko: apelacja może powołać fakt (datę, kwotę, nazwisko), którego w aktach nie ma. Sprawdzamy to **etapowo**:

1. wyciągamy z apelacji atomowe twierdzenia faktyczne,
2. dla każdego — po **opisach** dokumentów — wybieramy, w których plikach je sprawdzić, i weryfikujemy tylko w nich,
3. liczymy wskaźnik halucynacji.

> ⚠️ Najwolniejszy krok: opisuje wszystkie akta raz, potem sprawdza każde twierdzenie.

In [ ]:
from src.eval.grounding import evaluate_grounding

g = evaluate_grounding(appeal, docs)
print(f"Twierdzeń: {g.total}")
print(f"  potwierdzone:  {g.supported}")
print(f"  bez oparcia:   {g.unsupported}")
print(f"  sprzeczne:     {g.contradicted}")
print(f"\nWskaźnik halucynacji: {g.hallucination_rate:.0%}")

In [ ]:
print("Podejrzane twierdzenia (bez oparcia / sprzeczne z aktami):\n")
for r in g.results:
    if r.status != "supported":
        print(f"[{r.status}] {r.claim}")
        print(f"   sprawdzano w: {r.checked_files}")
        print(f"   ↳ {r.reasoning}\n")

## Podsumowanie — dwa mierniki, dwa ryzyka

| Miernik | Pyta | Łapie ryzyko |
|---------|------|--------------|
| **pokrycie** | czy poruszono wymagane zagadnienia? | apelacja **pomija** istotny zarzut |
| **ugruntowanie** | czy fakty pochodzą z akt? | apelacja **zmyśla** fakty (halucynacje) |

Wysokie pokrycie przy zmyślonych faktach to wciąż zła apelacja — dlatego potrzeba obu.

Porównanie podejść obok siebie: `python -m src.eval.compare` (po wygenerowaniu apelacji w każdym podejściu).